In [1]:
%load_ext cudf.pandas
%load_ext cuml.accel

import cudf
import pandas as pd
import cupy as cp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import re
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [37]:
df_train = pd.read_csv('datasets/Quora Questions Pair Dataset/train_preprocessed.csv')
print(type(df_train))
df_train.sample(3)

<class 'pandas.core.frame.DataFrame'>


,question1,question2,is_duplicate
257124,what are some examples of warm blooded and col...,what are the characteristics of warm blooded a...,0
97289,what is a year year swap rate used to benchmar...,what does a × at the moment receiver european ...,0
349031,how can i overcome my porn addiction,how do i get over my porn addiction,1


In [38]:
def tokenize(text):
    return text.split()

In [39]:
all_tokens = []
for col in ['question1', 'question2']:
    for row in df_train[col]:
        all_tokens.extend(tokenize(row))
print(f"Corpus size: {len(all_tokens)}")

Corpus size: 9016420


In [40]:
#get all the words with the number of times they repeat
counter = Counter(all_tokens)
counter

Counter({'the': 377839,
         'what': 324363,
         'is': 289059,
         'i': 223939,
         'how': 220801,
         'a': 212917,
         'in': 212427,
         'to': 206012,
         'do': 169383,
         'of': 159909,
         'are': 149286,
         'and': 134042,
         'can': 114060,
         'for': 104507,
         'you': 96125,
         'my': 86135,
         'why': 84021,
         'it': 71495,
         'best': 70586,
         'on': 60733,
         'does': 59436,
         'have': 44876,
         'or': 44718,
         'which': 43959,
         'be': 43843,
         'if': 43795,
         'some': 41677,
         'that': 40702,
         'with': 39727,
         'get': 39544,
         'should': 39541,
         'not': 38753,
         'an': 37008,
         'from': 35244,
         'your': 34025,
         'india': 29512,
         'will': 29419,
         's': 28597,
         'people': 26386,
         'when': 26215,
         'who': 26074,
         'like': 25634,
         'at': 2

In [41]:
# build vocab of all the words with 5 or more occurences
MIN_COUNT = 4
vocab = ['<pad>','<unk>'] + [w for w,c in counter.items() if c >= MIN_COUNT]
vocab

['<pad>',
 '<unk>',
 'what',
 'is',
 'the',
 'step',
 'by',
 'guide',
 'to',
 'invest',
 'in',
 'share',
 'market',
 'india',
 'story',
 'of',
 'kohinoor',
 'koh',
 'i',
 'noor',
 'diamond',
 'how',
 'can',
 'increase',
 'speed',
 'my',
 'internet',
 'connection',
 'while',
 'using',
 'a',
 'vpn',
 'why',
 'am',
 'mentally',
 'very',
 'lonely',
 'solve',
 'it',
 'which',
 'one',
 'dissolve',
 'water',
 'sugar',
 'salt',
 'methane',
 'and',
 'carbon',
 'di',
 'oxide',
 'astrology',
 'capricorn',
 'sun',
 'cap',
 'moon',
 'rising',
 'does',
 'that',
 'say',
 'about',
 'me',
 'should',
 'buy',
 'tiago',
 'be',
 'good',
 'geologist',
 'when',
 'do',
 'you',
 'use',
 'instead',
 'motorola',
 'company',
 'hack',
 'charter',
 'method',
 'find',
 'separation',
 'slits',
 'fresnel',
 'read',
 'youtube',
 'comments',
 'make',
 'physics',
 'easy',
 'learn',
 'was',
 'your',
 'first',
 'sexual',
 'experience',
 'like',
 'are',
 'laws',
 'change',
 'status',
 'from',
 'student',
 'visa',
 'green',


In [42]:
# generate index of all the words
word2idx = {w:i for i,w in enumerate(vocab)}
word2idx

{'<pad>': 0,
 '<unk>': 1,
 'what': 2,
 'is': 3,
 'the': 4,
 'step': 5,
 'by': 6,
 'guide': 7,
 'to': 8,
 'invest': 9,
 'in': 10,
 'share': 11,
 'market': 12,
 'india': 13,
 'story': 14,
 'of': 15,
 'kohinoor': 16,
 'koh': 17,
 'i': 18,
 'noor': 19,
 'diamond': 20,
 'how': 21,
 'can': 22,
 'increase': 23,
 'speed': 24,
 'my': 25,
 'internet': 26,
 'connection': 27,
 'while': 28,
 'using': 29,
 'a': 30,
 'vpn': 31,
 'why': 32,
 'am': 33,
 'mentally': 34,
 'very': 35,
 'lonely': 36,
 'solve': 37,
 'it': 38,
 'which': 39,
 'one': 40,
 'dissolve': 41,
 'water': 42,
 'sugar': 43,
 'salt': 44,
 'methane': 45,
 'and': 46,
 'carbon': 47,
 'di': 48,
 'oxide': 49,
 'astrology': 50,
 'capricorn': 51,
 'sun': 52,
 'cap': 53,
 'moon': 54,
 'rising': 55,
 'does': 56,
 'that': 57,
 'say': 58,
 'about': 59,
 'me': 60,
 'should': 61,
 'buy': 62,
 'tiago': 63,
 'be': 64,
 'good': 65,
 'geologist': 66,
 'when': 67,
 'do': 68,
 'you': 69,
 'use': 70,
 'instead': 71,
 'motorola': 72,
 'company': 73,
 'hack'

In [43]:
VOCAB_SIZE = len(vocab)
VOCAB_SIZE

32980

In [44]:
# tokenize a text --> consider upto first 25 words only
# if there are <25 words, append zeros at the end to make 25 words exactly
def encode(sent, max_words=25):
    # in word2idx.get(w,1) --> if w doesn't exist in vocab, return index 1 <'unk'>
    ids = [word2idx.get(w,1) for w in tokenize(sent)][:max_words]
    # append 0 at the end, 0=<'pad'> to make the length 25 exactly
    ids += [0] * (max_words - len(ids))
    return ids

In [45]:
MAX_WORDS = 30
df_train['question1'] = df_train['question1'].apply(lambda s: encode(s, MAX_WORDS))
df_train['question2'] = df_train['question2'].apply(lambda s: encode(s, MAX_WORDS))
df_train

,question1,question2,is_duplicate
0,"[2, 3, 4, 5, 6, 5, 7, 8, 9, 10, 11, 12, 10, 13...","[2, 3, 4, 5, 6, 5, 7, 8, 9, 10, 11, 12, 0, 0, ...",0
1,"[2, 3, 4, 14, 15, 16, 17, 18, 19, 20, 0, 0, 0,...","[2, 108, 817, 304, 4, 430, 199, 4275, 4, 16, 1...",0
2,"[21, 22, 18, 23, 4, 24, 15, 25, 26, 27, 28, 29...","[21, 22, 26, 24, 64, 1741, 6, 541, 162, 27538,...",0
3,"[32, 33, 18, 34, 35, 36, 21, 22, 18, 37, 38, 0...","[77, 4, 9718, 67, 1356, 1356, 3, 11189, 6, 0, ...",0
4,"[39, 40, 41, 10, 42, 1, 43, 44, 45, 46, 47, 48...","[39, 3012, 108, 5313, 10, 44, 42, 0, 0, 0, 0, ...",0
...,...,...,...
403963,"[21, 128, 2510, 94, 518, 10, 4, 16286, 367, 36...","[21, 128, 2510, 94, 518, 10, 8483, 367, 368, 1...",0
403964,"[68, 69, 1659, 518, 3, 425, 282, 1246, 0, 0, 0...","[3, 38, 1484, 57, 518, 3, 425, 282, 1246, 0, 0...",1
403965,"[2, 3, 40, 7787, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","[2, 3, 810, 7787, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0...",0
403966,"[2, 3, 4, 14380, 10367, 377, 15, 2188, 28, 240...","[18, 33, 2145, 3855, 1962, 1790, 1110, 18, 122...",0


In [46]:
# Skip-gram Word2Vec with negative sampling (using custom nn.Module on GPU)

WINDOW = 4
EMBED_DIM = 128
NEG_SAMPLES = 5

In [ ]:
def build_skipgram_pairs(token_list, window=WINDOW):
    pairs = []
    for tokens in token_list:
        